# FILE 1/3 — Luồng B, Chặng 1: Fine-tune COCO → CanTho (seed = 1712)

Notebook nay la **buoc dau tien** cua lo trinh tuan tu B, chay lai voi random seed moi
la **1712** (da chay truoc do voi seed 42 va 1407), theo yeu cau cua thay nham kiem chung
tinh on dinh cua ket luan qua 3 seed khac nhau.

**Vi tri trong toan bo quy trinh:**
- **FILE 1 (notebook nay):** COCO → CanTho → model B-CanTho (seed 1712)
- FILE 2: B-CanTho → HCMC → model B cuoi cung (seed 1712)
- FILE 3: train luồng A (COCO→HCMC) + danh gia ca A va B tren 3 tap test + bang so sanh 3 seed

**Luu y:**
- Chi doi **seed HUAN LUYEN** (seed=1712). Tap du lieu train/val/test GIU NGUYEN cach chia
  co san, de bao dam so sanh cong bang voi ket qua seed 42 va 1407.
- Model luu voi hau to `_seed1712`, KHONG de len ket qua cac seed truoc.
- Dung cau hinh Bang 3.4: imgsz=640, batch=16, epochs=100, patience=20, seed=1712.

**CHAY FILE NAY TRUOC. Sau khi xong moi chay FILE 2.**

> **LUU Y QUAN TRONG NEU CHAY BANG TAI KHOAN DUOC CHIA SE (vi du tai khoan cua Tai):**
> Thu muc "Do an AI giao thong" duoc share se nam o muc **"Shared with me"**, KHONG nam trong
> "My Drive", nen duong dan `/content/drive/MyDrive/Do an AI giao thong/...` se KHONG tim thay.
>
> **Cach xu ly (chi lam 1 lan, khong can sua code):**
> 1. Mo Google Drive tren web, vao muc **"Shared with me"**.
> 2. Tim thu muc **"Do an AI giao thong"**, chuot phai → **Organize** → **Add shortcut**.
> 3. Chon vi tri **"My Drive"** (thu muc goc) → bam **Add**.
>
> Sau buoc nay, moi duong dan trong notebook se hoat dong binh thuong.
> Cell "Buoc 3" ben duoi co dong kiem tra ton tai, neu bao False thi lam lai buoc tren.


## Buoc 1: Cai dat thu vien

In [1]:
!pip install ultralytics -q
print("Da cai dat xong ultralytics")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 8.5 MB/s eta 0:00:00
Da cai dat xong ultralytics


## Buoc 2: Ket noi Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Buoc 3: Khai bao duong dan va cau hinh

In [3]:
import os

SEED = 1712   # <=== SEED MOI theo yeu cau cua thay

# ===== MODEL KHOI TAO =====
BASELINE_COCO = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es/weights/best.pt"

# ===== DATASET (goc tren Drive) =====
DRIVE_DIR_CANTHO = "/content/drive/MyDrive/Do an AI giao thong/Dataset/CanTho_701515"

# ===== DATASET (copy ve local) =====
LOCAL_DIR_CANTHO = "/content/cantho_701515"

# ===== THU MUC LUU KET QUA =====
PROJECT_DIR = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline"
NAME_B_CANTHO = "finetune_cantho_seed1712"

# ===== CAU HINH HUAN LUYEN (Bang 3.4) =====
IMGSZ, BATCH, EPOCHS, PATIENCE, LR0 = 640, 16, 100, 20, 0.001

print("Kiem tra su ton tai:")
print("  Baseline COCO         :", os.path.exists(BASELINE_COCO))
print("  Dataset CanTho (Drive):", os.path.exists(DRIVE_DIR_CANTHO))
if not os.path.exists(BASELINE_COCO):
    print("\n[LOI] Khong tim thay duong dan. Neu dang dung tai khoan duoc chia se,")
    print("      hay tao shortcut thu muc 'Do an AI giao thong' vao My Drive (xem huong dan o dau notebook).")

Kiem tra su ton tai:
  Baseline COCO         : True
  Dataset CanTho (Drive): True


## Buoc 4: Copy dataset CanTho ve o cung local

Neu gap loi "Transport endpoint is not connected", chi can chay lai dung cell nay.

In [4]:
import shutil, time

if os.path.exists(LOCAL_DIR_CANTHO):
    shutil.rmtree(LOCAL_DIR_CANTHO)
t0 = time.time()
try:
    shutil.copytree(DRIVE_DIR_CANTHO, LOCAL_DIR_CANTHO)
    print(f"[OK] Da copy CanTho ve {LOCAL_DIR_CANTHO} ({time.time()-t0:.1f}s)")
except shutil.Error as e:
    print(f"[CANH BAO] {len(e.args[0])} file loi tam thoi khi copy.")

DATA_YAML_CANTHO_LOCAL = f"{LOCAL_DIR_CANTHO}/data.yaml"
print("Yaml CanTho (local) ton tai:", os.path.exists(DATA_YAML_CANTHO_LOCAL))

[OK] Da copy CanTho ve /content/cantho_701515 (55.8s)
Yaml CanTho (local) ton tai: True


## Buoc 5: Huan luyen chang 1 — COCO → CanTho (seed 1712)

Khoi tao tu Baseline COCO, fine-tune tren CanTho voi seed moi. Mat khoang 1 gio.
Nho bat GPU: Runtime → Change runtime type → T4 GPU.

In [5]:
from ultralytics import YOLO

model_b_cantho = YOLO(BASELINE_COCO)
results = model_b_cantho.train(
    data=DATA_YAML_CANTHO_LOCAL,
    epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    lr0=LR0, seed=SEED,
    project=PROJECT_DIR, name=NAME_B_CANTHO, exist_ok=True, verbose=True,
)
print("\nDa huan luyen xong chang 1 (COCO -> CanTho).")
BEST_B_CANTHO = os.path.join(PROJECT_DIR, NAME_B_CANTHO, "weights", "best.pt")
print("Model B-CanTho luu tai:", BEST_B_CANTHO)
print("Ton tai:", os.path.exists(BEST_B_CANTHO))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.147 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cantho_701515/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

## Buoc 6 (tuy chon): Danh gia nhanh chang 1 tren test CanTho

In [6]:
BEST_B_CANTHO = os.path.join(PROJECT_DIR, NAME_B_CANTHO, "weights", "best.pt")
m = YOLO(BEST_B_CANTHO).val(
    data=DATA_YAML_CANTHO_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name=NAME_B_CANTHO + "_eval_cantho", exist_ok=True,
)
print("===== CHANG 1 tren TEST CanTho =====")
print(f"mAP@0.5      : {float(m.box.map50):.4f}")
print(f"mAP@0.5:0.95 : {float(m.box.map):.4f}")
print(f"Precision    : {float(m.box.mp):.4f}")
print(f"Recall       : {float(m.box.mr):.4f}")
print("\n>>> DA XONG FILE 1. Tiep theo chay FILE 2.")

Ultralytics 8.4.147 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 46.1±13.7 MB/s, size: 40.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/.shortcut-targets-by-id/1WhcqICO5w6iVwbBcCYkPMXQf9-IvVZMs/Do an AI giao thong/Dataset/CanTho_701515/test/labels... 167 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 167/167 206.2it/s 0.8s
val: New cache created: /content/drive/.shortcut-targets-by-id/1WhcqICO5w6iVwbBcCYkPMXQf9-IvVZMs/Do an AI giao thong/Dataset/CanTho_701515/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 2.8it/s 3.9s
                   all        167        980      0.888      0.823      0.902      0.711
    